<a href="https://colab.research.google.com/github/ChandrimaGhorai/ScorePCR/blob/main/ScorePCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

class ScorePCREngine:
    def __init__(self, template, forward_primer, reverse_primer, na_concentration=50.0):
        self.template = template.strip().upper()
        self.forward_primer = forward_primer.strip().upper()
        self.reverse_primer = reverse_primer.strip().upper()
        self.na_concentration = na_concentration  # in mM, default 50 mM

    def validate_sequence(self, seq):
        valid_bases = set("ATGC")
        return all(base in valid_bases for base in seq)

    def calculate_gc(self, seq):
        if not seq:
            return 0.0
        gc_count = seq.count('G') + seq.count('C')
        return round((gc_count / len(seq)) * 100, 2)

    def calculate_tm(self, primer):
        a = primer.count('A')
        t = primer.count('T')
        g = primer.count('G')
        c = primer.count('C')
        length = len(primer)
        gc_pct = self.calculate_gc(primer)

        # 1. Basic Wallace Rule (For Short Oligos: 14-20 bases)
        if 14 <= length <= 20:
            tm = (2.0 * (a + t)) + (4.0 * (g + c))
            method = "Basic Wallace Rule: 2°C * (A + T) + 4°C * (G + C)"

        # 2. Salt-Adjusted Formula (For Mid-Length Sequences / General standard)
        else:
            # Converting Na+ concentration from mM to M for log calculation: [Na+] in M
            na_moles = self.na_concentration / 1000.0
            tm = 81.5 + (16.6 * math.log10(na_moles)) + (0.41 * gc_pct) - (600.0 / length)
            method = f"Salt-Adjusted Formula: 81.5 + 16.6 * log10([Na+]) + 0.41*(%GC) - (600 / N) with [Na+]={self.na_concentration}mM"

        return round(tm, 1), method

    def reverse_complement(self, seq):
        complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
        return "".join(complement.get(base, base) for base in reversed(seq))

    def evaluate_primer(self, name, primer):
        issues = []
        length = len(primer)
        gc = self.calculate_gc(primer)
        tm, tm_method = self.calculate_tm(primer)

        # Check length (17 to 23 bp)
        if not (17 <= length <= 23):
            issues.append(f"Length is {length} bp (Bad: must be between 17 and 23 bp)")

        # Check GC content (40% to 60%)
        if not (40.0 <= gc <= 60.0):
            issues.append(f"GC content is {gc}% (Bad: ideal range is 40% to 60%)")

        # Check Tm (45°C to 65°C)
        if not (45.0 <= tm <= 65.0):
            issues.append(f"Tm is {tm}°C (Bad: ideal range is 45°C to 65°C)")

        return {
            "name": name,
            "length": length,
            "gc": gc,
            "tm": tm,
            "tm_method": tm_method,
            "issues": issues,
            "is_good": len(issues) == 0
        }

    def run_analysis(self):
        print("\n=== ScorePCR In-Silico Analysis Report ===")

        if not (self.validate_sequence(self.template) and
                self.validate_sequence(self.forward_primer) and
                self.validate_sequence(self.reverse_primer)):
            print("Status: Failed | Error: Invalid DNA characters detected (allowed: A, T, G, C).")
            return

        print("Sequence Validation: PASSED")

        fp_eval = self.evaluate_primer("Forward Primer", self.forward_primer)
        rp_eval = self.evaluate_primer("Reverse Primer", self.reverse_primer)

        print(f"\n[Forward Primer Analysis]")
        print(f"Length: {fp_eval['length']} bp | GC Content: {fp_eval['gc']}% | Tm: {fp_eval['tm']}°C")
        print(f"  Model Used: {fp_eval['tm_method']}")
        if fp_eval['issues']:
            for issue in fp_eval['issues']:
                print(f"  ⚠️ WARNING: {issue}")
        else:
            print("  Status: Optimal")

        print(f"\n[Reverse Primer Analysis]")
        print(f"Length: {rp_eval['length']} bp | GC Content: {rp_eval['gc']}% | Tm: {rp_eval['tm']}°C")
        print(f"  Model Used: {rp_eval['tm_method']}")
        if rp_eval['issues']:
            for issue in rp_eval['issues']:
                print(f"  ⚠️ WARNING: {issue}")
        else:
            print("  Status: Optimal")

        # Binding sites detection (using reverse complement for reverse primer search)
        fp_pos = self.template.find(self.forward_primer)
        rc_reverse_primer = self.reverse_complement(self.reverse_primer)
        rp_pos = self.template.find(rc_reverse_primer)

        print(f"\n[Binding-Site Detection]")
        if fp_pos != -1:
            print(f"Forward Primer Binding Site: Index {fp_pos} to {fp_pos + len(self.forward_primer) - 1}")
        else:
            print("Forward Primer Binding Site: Not found on template")

        if rp_pos != -1:
            print(f"Reverse Primer Binding Site (RC matched): Index {rp_pos} to {rp_pos + len(self.reverse_primer) - 1}")
        else:
            print("Reverse Primer Binding Site: Not found on template")

        # In-Silico PCR & Amplicon Length Calculation
        if fp_pos != -1 and rp_pos != -1 and fp_pos < rp_pos:
            amplicon_size = (rp_pos + len(rc_reverse_primer)) - fp_pos
            print(f"\n[In-Silico PCR Simulation]")
            print(f"Total Amplicon Length: {amplicon_size} bp")

            if fp_eval['is_good'] and rp_eval['is_good']:
                print("Final Result: Good")
                print("Reasoning: Primers meet all length, GC%, and Tm criteria, successfully yielding a target amplicon.")
            else:
                print("Final Result: Needs Attention")
                print("Reasoning: Amplicon is generated, but one or more primers failed ideal thresholds.")
        else:
            print("\n[In-Silico PCR Simulation]")
            print("Final Result: Poor")
            print("Troubleshooting: Primers do not face each other correctly, overlap, or binding sites are missing on the template.")

if __name__ == "__main__":
    print("--- ScorePCR Interactive Tool ---")
    user_template = input("Enter DNA Template Sequence: ")
    user_fp = input("Enter Forward Primer Sequence: ")
    user_rp = input("Enter Reverse Primer Sequence: ")

    pcr_tool = ScorePCREngine(user_template, user_fp, user_rp)
    pcr_tool.run_analysis()

--- ScorePCR Interactive Tool ---
Enter DNA Template Sequence: AATGGAAAAATTCGAAGGGTATTCAGAAAAACAAAAATCTCGTCAACAATACTTCGTCTACCCACTTCTCTTTCAGGAATATATTTATGCATTTGCCCATGATTATGGATTAAATGGTTCCGAACCTGTGGAAATTATTGGGTGTAATAACAAGAAATTTAGTTCACTACTTGTGAAACGTTTAATTATTCGAATGTATCAGCAGAATTTTTGGATTAATTCGGTTAATCATCCTAACCAAGATCGATTGTTGGATCACAGCAATTATTTTTATTCTGAGTTTTATTCTCAGATTCTATCTGAGGGGTTTGCGATCGTTGTAGAAATCCCATTCTCGCTAGGAGAACTATCTTGTCCGGAAGAAAAAGAAATACCAAAGTTTCAAAATTTACAATCTATTCATTCAATATTTCCTTTTTTAGAAGACAAATTTTTGCATTTACATTATCTATCACATCTAGAAATACCCTATCCTATCCATTTGGAAATCTTGGTTCAACTCCTTGAATACCGGATCCAAGATGTTCCATCTTTGCATTTATTGCGATTC
Enter Forward Primer Sequence: CGAACCTGTGGAAATTATTGGG
Enter Reverse Primer Sequence: GGAACATCTTGGATCCGGTATT

=== ScorePCR In-Silico Analysis Report ===
Sequence Validation: PASSED

[Forward Primer Analysis]
Length: 22 bp | GC Content: 45.45% | Tm: 51.3°C
  Model Used: Salt-Adjusted Formula: 81.5 + 16.6 * log10([Na+]) + 0.41*(%GC) - (600 / N) with [Na+]=50.0mM
  Status: Optimal

[Reve